# NumPy

## A Particle Model of an Ideal Gas
### Tel-Aviv University / 0509-1820 / Fall 2025-2026

## Recitation Goals

- Represent scientific data with NumPy arrays.
- Use vectorized calculations instead of handling one measurement at a time.
- Practice `axis`, Boolean masks, reductions, and broadcasting in a physical context.
- Build a simple particle model of an ideal gas and compare its trends with the gas laws.

In the next recitation we will use Matplotlib to plot the simulation results.

## Importing NumPy

NumPy is usually imported with the short name `np`.

In [ ]:
import numpy as np

## Warm-up: Chemical Measurements as Arrays

Suppose we measured absorbance for several known solution concentrations. Instead of storing each measurement in a separate variable, we store the whole series in one array.

In [ ]:
concentration_mM = np.array([0.00, 0.20, 0.40, 0.60, 0.80, 1.00])
absorbance = np.array([0.01, 0.15, 0.30, 0.44, 0.61, 0.74])

print(concentration_mM)
print(absorbance)
print(type(absorbance), absorbance.shape)

### Elementwise Operations

An operation between an array and a number is applied to every element. For example, converting from mM to M:

In [ ]:
concentration_M = concentration_mM / 1000
print(concentration_M)

### Boolean Masks

A Boolean mask is an array of `True`/`False` values. We can use it to select only measurements that satisfy a condition.

In [ ]:
valid = absorbance > 0.05
print(valid)
print(concentration_mM[valid])
print(absorbance[valid])

### Reductions: Mean, Standard Deviation, and Extremes

Functions such as `np.mean`, `np.std`, `np.min`, and `np.max` reduce an array to a smaller result, often a single number. These operations are common in measurement analysis.

In [ ]:
print('mean absorbance:', np.mean(absorbance))
print('std absorbance:', np.std(absorbance))
print('max absorbance:', np.max(absorbance))

## From Measurements to a Model: An Ideal Gas in a 2D Box

We will build a simple model: point particles move in a square box and collide elastically with the walls. This kind of model can be treated theoretically on paper, but a computer simulation is more general: if we want to, it is straightforward to include interactions between particles or a more elaborately shaped box.

We use dimensionless units:

- Each particle has mass `m = 1`.
- Boltzmann's constant is `k_B = 1`.
- Temperature is defined from the average kinetic energy.

## Random Initial Conditions

The lecture notes already used `np.random` to create random arrays. Here we use the newer NumPy random-number generator:

```python
rng = np.random.default_rng(7)
```

The number `7` is the seed. It makes the random choices reproducible: when we rerun the notebook, we get the same example again. We will use two kinds of random numbers:

- `rng.uniform(low, high, size=...)` for random positions inside the box.
- `rng.normal(mean, std, size=...)` for random velocity components.

In [ ]:
rng = np.random.default_rng(7)

print(rng.uniform(0, 10, size=4))
print(rng.normal(0, 1, size=4))

## Representing Positions and Velocities

We represent `N` particles in two dimensions. The positions are stored in an array of shape `(N, 2)`: one row per particle, one column for `x`, and one column for `y`.

For example, an array with 3 particles has this structure:

| row | meaning | column 0 | column 1 |
|---|---|---:|---:|
| 0 | particle 0 | $x_0$ | $y_0$ |
| 1 | particle 1 | $x_1$ | $y_1$ |
| 2 | particle 2 | $x_2$ | $y_2$ |

Velocities use the same structure, with columns for $v_x$ and $v_y$.

In [ ]:
example_positions = np.array([
    [2.0, 8.0],
    [4.5, 1.0],
    [9.0, 5.5],
])

print(example_positions)
print(example_positions.shape)
print('x coordinates:', example_positions[:, 0])
print('y coordinates:', example_positions[:, 1])

Now create a small random system. We start with only 5 particles so that the arrays are easy to print.

In [ ]:
N = 5
box_length = 10.0

positions = rng.uniform(0, box_length, size=(N, 2))
velocities = rng.normal(0, 1, size=(N, 2))

print('positions shape:', positions.shape)
print(positions)
print('velocities shape:', velocities.shape)
print(velocities)

### `axis`: Per Particle or Per Column

For a two-dimensional array, `axis` controls the direction of the calculation. This appeared in the lecture notes, but it is worth slowing down here.

If we want one result per particle, we combine the two columns in each row, so we use `axis=1`.

In [ ]:
small_velocities = np.array([
    [3.0, 4.0],
    [1.0, 2.0],
])

squared_components = small_velocities**2
speed_squared_manual = squared_components[:, 0] + squared_components[:, 1]
speed_squared_axis = np.sum(squared_components, axis=1)

print(squared_components)
print(speed_squared_manual)
print(speed_squared_axis)

Now apply the same calculation to the velocity array from the simulation.

In [ ]:
speed_squared = np.sum(velocities**2, axis=1)
speeds = np.sqrt(speed_squared)

print(speed_squared)
print(speeds)

### Temperature from Kinetic Energy

In a two-dimensional model with `m = 1` and `k_B = 1`, the average kinetic energy per particle is:

$$\langle E_k \rangle = \frac{1}{2}\langle v_x^2 + v_y^2 \rangle$$

The corresponding temperature is approximately:

$$T = \frac{\langle v_x^2 + v_y^2 \rangle}{2}$$

In [ ]:
def temperature_from_velocities(velocities):
    speed_squared = np.sum(velocities**2, axis=1)
    return np.mean(speed_squared) / 2

print(temperature_from_velocities(velocities))

## Rescaling Velocities to a Target Temperature

If we multiply all velocities by the same factor, the kinetic energy changes by the square of that factor. We can use this to prepare an initial state with a desired temperature.

In [ ]:
def rescale_temperature(velocities, target_temperature):
    current_temperature = temperature_from_velocities(velocities)
    scale = np.sqrt(target_temperature / current_temperature)
    return velocities * scale

velocities_T2 = rescale_temperature(velocities, target_temperature=2.0)
print(temperature_from_velocities(velocities_T2))

## One Simulation Step

At each time step:

1. Update position using `position = position + velocity * dt`.
2. Check which particles crossed each wall.
3. Move those particles back into the box and reverse the velocity component that hit the wall.

The code below is intentionally explicit. It handles the `x` walls and the `y` walls separately instead of using a shorter two-dimensional trick. That makes it easier to see which velocity component changes sign.

### Momentum Transfer at a Wall

When a particle hits a vertical wall, its $v_x$ changes from $v_x$ to $-v_x$. With mass 1, the magnitude of the momentum change is:

$$|\Delta p_x| = 2|v_x|$$

The same idea applies to $v_y$ at the horizontal walls. We sum these momentum transfers to estimate pressure later.

In [ ]:
def step(positions, velocities, box_length, dt):
    new_positions = positions + velocities * dt
    new_velocities = velocities.copy()
    momentum_to_walls = 0.0

    hit_left = new_positions[:, 0] < 0
    hit_right = new_positions[:, 0] > box_length
    hit_x_wall = hit_left | hit_right

    momentum_to_walls += np.sum(2 * np.abs(new_velocities[hit_x_wall, 0]))
    new_velocities[hit_x_wall, 0] = -new_velocities[hit_x_wall, 0]

    new_positions[hit_left, 0] = -new_positions[hit_left, 0]
    new_positions[hit_right, 0] = 2 * box_length - new_positions[hit_right, 0]

    hit_bottom = new_positions[:, 1] < 0
    hit_top = new_positions[:, 1] > box_length
    hit_y_wall = hit_bottom | hit_top

    momentum_to_walls += np.sum(2 * np.abs(new_velocities[hit_y_wall, 1]))
    new_velocities[hit_y_wall, 1] = -new_velocities[hit_y_wall, 1]

    new_positions[hit_bottom, 1] = -new_positions[hit_bottom, 1]
    new_positions[hit_top, 1] = 2 * box_length - new_positions[hit_top, 1]

    return new_positions, new_velocities, momentum_to_walls

### Quick Check of One Step

Create one particle near a wall and verify that the velocity component in the collision direction changes sign.

In [ ]:
test_positions = np.array([[9.8, 5.0]])
test_velocities = np.array([[3.0, 0.0]])

new_pos, new_vel, impulse = step(test_positions, test_velocities, box_length=10.0, dt=0.2)
print(new_pos)
print(new_vel)
print('momentum transferred:', impulse)

## Running a Simulation

Pressure is calculated from the momentum transferred to the walls per unit time and per unit wall length. This is a simplified two-dimensional version of the physical idea: pressure comes from particle collisions with the container walls.

In [ ]:
def simulate_gas(N=200, box_length=10.0, target_temperature=1.0, steps=4000, dt=0.01, seed=1):
    rng = np.random.default_rng(seed)
    positions = rng.uniform(0, box_length, size=(N, 2))
    velocities = rng.normal(0, 1, size=(N, 2))
    velocities = rescale_temperature(velocities, target_temperature)

    total_momentum_to_walls = 0.0
    temperatures = np.zeros(steps)

    for i in range(steps):
        positions, velocities, impulse = step(positions, velocities, box_length, dt)
        total_momentum_to_walls += impulse
        temperatures[i] = temperature_from_velocities(velocities)

    total_time = steps * dt
    perimeter = 4 * box_length
    pressure = total_momentum_to_walls / (total_time * perimeter)
    area = box_length**2

    return {
        'N': N,
        'box_length': box_length,
        'area': area,
        'target_temperature': target_temperature,
        'mean_temperature': np.mean(temperatures),
        'pressure': pressure,
        'pressure_area': pressure * area,
        'positions': positions,
        'velocities': velocities,
        'temperatures': temperatures,
    }

### Basic Run

For an ideal gas, $PV = Nk_BT$. In this two-dimensional model we use area instead of volume, and we chose $k_B = 1$, so we expect approximately:

$$P \cdot A \approx N \cdot T$$

In [ ]:
result = simulate_gas(N=300, box_length=10.0, target_temperature=1.5, seed=4)

print('mean T:', result['mean_temperature'])
print('P:', result['pressure'])
print('P*A:', result['pressure_area'])
print('N*T:', result['N'] * result['mean_temperature'])
print('ratio (P*A)/(N*T):', result['pressure_area'] / (result['N'] * result['mean_temperature']))

## Experiment 1: Changing Temperature

Keep the number of particles and the box size fixed, and change only the temperature. According to the gas law, pressure should increase approximately linearly with temperature.

In [ ]:
temperatures_to_test = np.array([0.5, 1.0, 1.5, 2.0, 3.0])
pressure_by_temperature = np.zeros(temperatures_to_test.size)

for i in range(temperatures_to_test.size):
    T = temperatures_to_test[i]
    res = simulate_gas(N=300, box_length=10.0, target_temperature=T, seed=10 + i)
    pressure_by_temperature[i] = res['pressure']

print('T values:', temperatures_to_test)
print('P values:', pressure_by_temperature)
print('P/T:', pressure_by_temperature / temperatures_to_test)

## Experiment 2: Changing the Number of Particles

Now keep temperature and box size fixed, and change the number of particles. According to the gas law, pressure should increase approximately linearly with `N`.

In [ ]:
particle_counts = np.array([100, 200, 300, 400, 600])
pressure_by_N = np.zeros(particle_counts.size)

for i in range(particle_counts.size):
    N_value = int(particle_counts[i])
    res = simulate_gas(N=N_value, box_length=10.0, target_temperature=1.5, seed=30 + i)
    pressure_by_N[i] = res['pressure']

print('N values:', particle_counts)
print('P values:', pressure_by_N)
print('P/N:', pressure_by_N / particle_counts)

## Experiment 3: Changing Box Area

Now keep `N` and temperature fixed, and change the side length of the box. The area is `box_length**2`. According to the gas law, pressure should decrease as area increases.

In [ ]:
box_lengths = np.array([8.0, 10.0, 12.0, 14.0])
pressure_by_area = np.zeros(box_lengths.size)
areas = box_lengths**2

for i in range(box_lengths.size):
    L = box_lengths[i]
    res = simulate_gas(N=300, box_length=L, target_temperature=1.5, seed=50 + i)
    pressure_by_area[i] = res['pressure']

print('areas:', areas)
print('P values:', pressure_by_area)
print('P*A:', pressure_by_area * areas)

## Class Exercise: Checking the Gas Law

Complete the function below. It should receive one simulation result and return the ratio:

$$\frac{P \cdot A}{N \cdot T}$$

For an ideal model with enough simulation steps, this ratio should be close to 1.

In [ ]:
def ideal_gas_ratio(result):
    # TODO: replace None with your calculation
    return None

# Example check:
# ideal_gas_ratio(result)

## Class Exercise: A Mask for Fast Particles

Complete the code so that it calculates what percentage of particles at the end of the simulation are moving faster than `speed_limit`.

In [ ]:
speed_limit = 2.5
final_velocities = result['velocities']

# TODO:
# 1. calculate the speed of each particle
# 2. create a Boolean mask for particles above speed_limit
# 3. calculate the percentage of particles that match the mask

fast_percentage = None
print(fast_percentage)

## Class Exercise: Broadcasting for a Prediction Table

Broadcasting was introduced in the lecture notes. Here is a reminder in the context of the ideal-gas prediction $P = NT/A$.

We want all combinations of several `N` values and several `T` values. `N_values[:, None]` turns `N_values` from shape `(3,)` into shape `(3, 1)`, a column. NumPy can then broadcast this column against the temperature row.

In [ ]:
N_values = np.array([100, 200, 300])
T_values = np.array([1.0, 2.0, 3.0])
A = 100.0

N_column = N_values[:, None]
print('N_values shape:', N_values.shape)
print('N_column shape:', N_column.shape)
print('T_values shape:', T_values.shape)

predicted_pressure = N_column * T_values / A
print(predicted_pressure)
print(predicted_pressure.shape)

## Summary Task

Choose one experiment above and change one additional parameter:

- the number of simulation steps,
- the time step `dt`,
- the number of particles,
- the temperature.

Check whether the ratio $PA/(NT)$ moves closer to 1 or farther from 1. In the next recitation, we will plot these trends and ask which plot is appropriate for each scientific question.

In [ ]:
# Space for your experiment

## Summary

In this recitation we used NumPy to move from simple measurements to a small physical model. We practiced arrays, elementwise operations, masks, `axis`, reductions, and broadcasting. The goal was not to build a full molecular dynamics simulation, but to see how arrays let us formulate and test a chemical-physical question computationally.